In [ ]:
#| default_exp domain

# Domain

> Johnny Decimal data models and JSON config loader for domain/area/category hierarchy.

In [ ]:
#| export
from pydantic import BaseModel, Field
from pathlib import Path
import json

In [ ]:
#| export
class JDItem(BaseModel):
    """A single item within a Johnny Decimal category."""
    id: str
    name: str

class JDCategory(BaseModel):
    """A category (e.g. '13 Shell and terminals') within a JD area."""
    id: str
    name: str
    description: str = ""
    tags: list[str] = []
    items: list[JDItem] = []

class JDArea(BaseModel):
    """An area (e.g. '10-19 Programming Languages') containing categories."""
    id: str
    name: str
    description: str = ""
    tags: list[str] = []
    categories: list[JDCategory] = []

class JDDomain(BaseModel):
    """A top-level domain (e.g. 'd6 Programming-Development') containing areas."""
    name: str
    domain_id: str = ""
    areas: list[JDArea] = []

    def model_post_init(self, __context):
        if not self.domain_id: self.domain_id = self.name.split()[0]

    def get_category(self, cat_id: str) -> JDCategory | None:
        """Find a category by its ID across all areas."""
        for area in self.areas:
            for cat in area.categories:
                if cat.id == cat_id: return cat
        return None

    @property
    def all_categories(self) -> list[JDCategory]:
        """Flat list of all categories across all areas."""
        return [cat for area in self.areas for cat in area.categories]

In [ ]:
#| export
def load_domains(config_dir: Path) -> list[JDDomain]:
    """Load all JD domain configs from JSON files in `config_dir`."""
    domains = []
    for f in sorted(config_dir.glob("d*.json")):
        data = json.loads(f.read_text(encoding="utf-8"))
        domains.append(JDDomain(**data))
    return domains

In [ ]:
#| export
def category_path(domain: JDDomain, category: JDCategory) -> str:
    """Build the filesystem path segment for a category: '{domain_id}/{cat_id} {cat_name}'."""
    return f"{domain.domain_id}/{category.id} {category.name}"

## Tests

In [ ]:
# Locate the repo's config/ from wherever the notebook is executed. Was hardcoded to
# "/home/jelle/code/hfm-tools/config" (lowercase "code"); the repo lives at ~/Code, so on a
# case-sensitive filesystem the path did not exist, zero domains loaded, and these tests failed.
def _repo_config_dir() -> Path:
    for base in [Path.cwd(), *Path.cwd().parents]:
        cand = base / "config"
        if cand.is_dir() and any(cand.glob("d*.json")): return cand
    raise FileNotFoundError("config/ with d*.json not found above cwd")

from fastcore.test import *

config_dir = _repo_config_dir()
domains = load_domains(config_dir)

# Should load all 3 domain files
test_eq(len(domains), 3)

# Check domain IDs are extracted correctly
domain_ids = {d.domain_id for d in domains}
test_eq(domain_ids, {"d1", "d5", "d6"})

# Check d6 has expected structure
d6 = next(d for d in domains if d.domain_id == "d6")
test_eq(d6.name, "d6 Programming-Development")
assert len(d6.areas) > 0

# Check category lookup
cat34 = d6.get_category("34")
assert cat34 is not None
test_eq(cat34.name, "Cloud platforms")

# Check all_categories
all_cats = d6.all_categories
assert len(all_cats) > 10

print(f"\u2713 Loaded {len(domains)} domains")
for d in domains:
    print(f"  {d.domain_id}: {d.name} ({len(d.all_categories)} categories)")

In [ ]:
test_eq(category_path(d6, cat34), "d6/34 Cloud platforms")

# Test d1 domain
d1 = next(d for d in domains if d.domain_id == "d1")
cat11 = d1.get_category("11")
assert cat11 is not None
test_eq(category_path(d1, cat11), "d1/11 Gezin")

print("\u2713 All domain tests passed")